In [5]:
import logging
from pathlib import Path

LOG_DIR = Path("logs")
LOG_DIR.mkdir(exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(message)s",
    handlers=[
        logging.FileHandler(LOG_DIR / "rag_pipeline.log", encoding="utf-8"),
        logging.StreamHandler()
    ]
)

logger = logging.getLogger("RAG")

In [6]:
from sentence_transformers import SentenceTransformer, CrossEncoder
from pgvector.psycopg2 import register_vector
import psycopg2
import ollama
from dotenv import load_dotenv
load_dotenv()

from langsmith import traceable
HOST = "localhost"
PORT = 5434
USER = "admin"
PASSWORD = "pass123"
DATABASE = "RAG_POC"

TABLE = "rag_chunks"

EMBED_MODEL = "BAAI/bge-m3"
RERANK_MODEL = "BAAI/bge-reranker-v2-m3"
OLLAMA_MODEL = "qwen3:8b"
print("Loading models...")

embed_model = SentenceTransformer(EMBED_MODEL)

reranker = CrossEncoder(RERANK_MODEL)
@traceable(name="RAG Retrieval", run_type="chain")
def data_retrival(
    SEARCH_QUERY,
    QUESTION,
    RULES,
    TOP_K_RETRIEVAL=10,
    TOP_K_FINAL=5,
):

    # ==========================================================
    # Embedding
    # ==========================================================

    @traceable(name="Embedding", run_type="embedding")
    def create_embedding(text):
        return embed_model.encode(
            text,
            normalize_embeddings=True
        ).tolist()

    query_embedding = create_embedding(SEARCH_QUERY)
    embedding_str = "[" + ",".join(map(str, query_embedding)) + "]"

    # ==========================================================
    # Vector Search
    # ==========================================================

    @traceable(name="Vector Search", run_type="retriever")
    def semantic_search(embedding, top_k):

        conn = psycopg2.connect(
            host=HOST,
            port=PORT,
            user=USER,
            password=PASSWORD,
            dbname=DATABASE,
        )

        register_vector(conn)

        cur = conn.cursor()

        sql = f"""
        SELECT
            page_content,
            source,
            metadata,
            embedding <=> %s::vector AS distance
        FROM {TABLE}
        ORDER BY distance
        LIMIT %s;
        """

        cur.execute(sql, (embedding, top_k))

        rows = cur.fetchall()

        cur.close()
        conn.close()

        return rows

    rows = semantic_search(
        embedding_str,
        TOP_K_RETRIEVAL
    )

    if not rows:
        logger.info("No documents found.")
        return "No documents found."

    logger.info("=" * 80)
    logger.info("TOP VECTOR SEARCH RESULTS")
    logger.info("=" * 80)

    for i, row in enumerate(rows, start=1):

        page_content, source, metadata, distance = row

        logger.info(f"Rank {i}")
        logger.info(f"Distance : {distance:.5f}")
        logger.info(page_content[:300])

    # ==========================================================
    # Reranker
    # ==========================================================

    @traceable(name="CrossEncoder Reranker", run_type="reranker")
    def rerank_documents(query, rows):

        rerank_query = f"""
Search Query:
{SEARCH_QUERY}

User Question:
{QUESTION}
"""

        pairs = [
            (rerank_query, row[0])
            for row in rows
        ]

        scores = reranker.predict(pairs)

        reranked = []

        for row, score in zip(rows, scores):

            page_content, source, metadata, distance = row

            reranked.append(
                {
                    "page_content": page_content,
                    "source": source,
                    "metadata": metadata,
                    "distance": distance,
                    "rerank_score": float(score),
                }
            )

        reranked.sort(
            key=lambda x: x["rerank_score"],
            reverse=True,
        )

        return reranked[:TOP_K_FINAL]

    reranked = rerank_documents(
        SEARCH_QUERY,
        rows,
    )

    logger.info("=" * 80)
    logger.info("TOP RERANKED CHUNKS")
    logger.info("=" * 80)

    for i, doc in enumerate(reranked, start=1):

        logger.info(f"Rank {i}")
        logger.info(f"Distance : {doc['distance']:.5f}")
        logger.info(f"Reranker : {doc['rerank_score']:.5f}")

    # ==========================================================
    # Prompt Builder
    # ==========================================================

    @traceable(name="Prompt Builder", run_type="prompt")
    def build_prompt(context, question, rules):

        STANDARD_RULES = [
            "Use ONLY the supplied context.",
            "Do NOT use outside knowledge.",
            "Return only the requested information.",
            "Do not explain unless requested.",
            "Do not summarize unless requested.",
            "If the answer appears explicitly in the context, return it exactly.",
            "If the context partially answers the question, return the available information.",
            "Only reply 'Data is not available.' when no relevant information exists.",
        ]

        all_rules = STANDARD_RULES + rules

        rule_text = "\n".join(
            f"{i}. {r}"
            for i, r in enumerate(all_rules, 1)
        )

        prompt = f"""
You are a RAG assistant.

Answer ONLY from the supplied context.

Context:
{context}

Question:
{question}

Rules:
{rule_text}

Answer:
"""

        return prompt

    context = "\n\n".join(
        doc["page_content"]
        for doc in reranked
    )

    prompt = build_prompt(
        context,
        QUESTION,
        RULES,
    )

    # ==========================================================
    # LLM
    # ==========================================================

    @traceable(name="Answer Generation", run_type="llm")
    def generate_answer(prompt):

        return ollama.chat(
            model=OLLAMA_MODEL,
            messages=[
                {
                    "role": "user",
                    "content": prompt,
                }
            ],
        )

    response = generate_answer(prompt)

    answer = response["message"]["content"]

    if "</think>" in answer:
        answer = answer.split("</think>", 1)[1].strip()

    logger.info("=" * 80)
    logger.info("FINAL ANSWER")
    logger.info("=" * 80)
    logger.info(answer)

    logger.info("=" * 80)
    logger.info("RETRIEVAL STATISTICS")
    logger.info("=" * 80)
    logger.info(f"Vector Search Top-K : {TOP_K_RETRIEVAL}")
    logger.info(f"Reranker Top-K      : {TOP_K_FINAL}")
    logger.info(f"Context Chunks      : {len(reranked)}")
    logger.info(f"Prompt Characters   : {len(prompt)}")

    with open("../console.md", "w", encoding="utf-8") as f:
        f.write(answer)

    return answer

2026-07-27 16:48:51,483 | INFO     | No device provided, using cpu


Loading models...


2026-07-27 16:48:52,351 | INFO     | HTTP Request: GET https://huggingface.co/api/agent-harnesses "HTTP/1.1 200 OK"
2026-07-27 16:48:52,603 | INFO     | HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-07-27 16:48:52,606 | WARNING  | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-07-27 16:48:52,969 | INFO     | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-m3/5617a9f61b028005a4858fdac845db406aefb181/modules.json "HTTP/1.1 200 OK"
2026-07-27 16:48:53,273 | INFO     | HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-07-27 16:48:53,285 | INFO     | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-m3/5617a9f61b028005a4858fdac845db406aefb181/config_sentence_transformers.json "HTTP/1

In [7]:
import json
import ollama

@traceable(name="Question Planner", run_type="llm"
)
def prepare_question(question):
    """
    Generate:
        1. Optimized semantic search query
        2. Question-specific answer rules
    """

    planner_prompt = f"""
You are an expert Query Planner for a Retrieval-Augmented Generation (RAG) system.

Your job is NOT to answer the user's question.

Your ONLY job is to prepare retrieval instructions.

=====================================================================

USER QUESTION

{question}

=====================================================================

TASK 1

Generate the BEST semantic search query.

Guidelines:

- Preserve technical terminology exactly.
- Preserve product names.
- Preserve menu names.
- Preserve UI labels.
- Preserve figure names.
- Preserve table names.
- Preserve chapter names.
- Preserve command names.
- Preserve important noun phrases.
- Remove conversational words.
- Remove answer formatting instructions.
- Keep the search query concise.
- Maximum 12 words.

=====================================================================

TASK 2

Generate ONLY question-specific answer rules.

Examples of GOOD rules:


- Preserve the original numbering.
- Return the complete procedure.
- Preserve the original wording.

DO NOT generate:

- Generic RAG rules.
- Explanations.
- New questions.
- Configuration steps.
- Hallucinated information.

=====================================================================

Return ONLY valid JSON.

Expected JSON format:

{{
    "search_query": "...",
    "rules": [
        "...",
        "...",
        "..."
    ]
}}

Example

User Question:

Give me the table of Hot Keys in the Operator Workplace.
Print the output in table format.
Skip first 7 rows.

Expected Output:

{{
    "search_query": "Hot Keys, Operator Workplace",

    "rules": [
        "Return the answer in table format.",
        "Skip the first 7 rows.",
        "Preserve the original row order."
    ]
}}

IMPORTANT

Return ONLY JSON.

Do NOT return Markdown.

Do NOT wrap the JSON inside ```json.

Do NOT explain your reasoning.

Do NOT include <think>.
"""

    response = ollama.chat(
        model=OLLAMA_MODEL,
        messages=[
            {
                "role": "user",
                "content": planner_prompt
            }
        ]
    )

    text = response["message"]["content"].strip()

    # ----------------------------------------------------------
    # Remove <think>...</think> (Qwen3 sometimes generates this)
    # ----------------------------------------------------------

    if "</think>" in text:
        text = text.split("</think>", 1)[1].strip()

    # ----------------------------------------------------------
    # Remove Markdown code fences
    # ----------------------------------------------------------

    if text.startswith("```"):
        text = text.replace("```json", "")
        text = text.replace("```", "")
        text = text.strip()

    # ----------------------------------------------------------
    # Parse JSON
    # ----------------------------------------------------------

    try:
        result = json.loads(text)

    except json.JSONDecodeError:

        logger.info("=" * 80)
        logger.info("INVALID JSON RETURNED BY PLANNER")
        logger.info("=" * 80)
        logger.info(text)

        raise

    # ----------------------------------------------------------
    # Print Planner Output
    # ----------------------------------------------------------

    logger.info("\n" + "=" * 80)
    logger.info("SEARCH QUERY")
    logger.info("=" * 80)
    logger.info(result["search_query"])

    logger.info("\n" + "=" * 80)
    logger.info("QUESTION SPECIFIC RULES")
    logger.info("=" * 80)

    for rule in result["rules"]:
        logger.info(f"- {rule}")

    logger.info("-------")

    return result["search_query"], result["rules"]

In [8]:
from langsmith import traceable

@traceable(name="Complete RAG Pipeline", run_type="chain")
def rag_pipeline(question):

    search_query, rules = prepare_question(question)

    answer = data_retrival(
        SEARCH_QUERY=search_query,
        QUESTION=question,
        RULES=rules
    )

    return {
        "QUESTION": question,
        "SEARCH_QUERY": search_query,
        "RULES": rules,
        "ANSWER": answer
    }


QUESTION = """
which figure number should i refer application bar configure for Alarm Logger Manager?
"""

result = rag_pipeline(QUESTION)

print(result["ANSWER"])

logger.info("=" * 80)
logger.info("FINAL ANSWER")
logger.info("=" * 80)
logger.info(result["ANSWER"])

logger.info("=" * 80)
logger.info("SEARCH QUERY")
logger.info("=" * 80)
logger.info(result["SEARCH_QUERY"])

logger.info("=" * 80)
logger.info("QUESTION SPECIFIC RULES")
logger.info("=" * 80)

for rule in result["RULES"]:
    logger.info(f"- {rule}")

with open("../console.md", "w", encoding="utf-8") as f:
    f.write(result["ANSWER"])

logger.info("=" * 80)
logger.info("Pipeline Completed")
logger.info("=" * 80)

2026-07-27 16:49:44,021 | INFO     | HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-07-27 16:49:44,023 | INFO     | 
2026-07-27 16:49:44,024 | INFO     | SEARCH QUERY
2026-07-27 16:49:44,024 | INFO     | ================================================================================
2026-07-27 16:49:44,024 | INFO     | figure number application bar configure Alarm Logger Manager
2026-07-27 16:49:44,025 | INFO     | 
2026-07-27 16:49:44,025 | INFO     | QUESTION SPECIFIC RULES
2026-07-27 16:49:44,025 | INFO     | ================================================================================
2026-07-27 16:49:44,026 | INFO     | - Preserve the original numbering.
2026-07-27 16:49:44,027 | INFO     | - Return the complete figure number.
2026-07-27 16:49:44,027 | INFO     | -------
Batches: 100%|██████████| 1/1 [00:00<00:00,  1.16it/s]
2026-07-27 16:49:45,157 | INFO     | ================================================================================
2026-07-2

Figure 92
